In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import sum
from pyspark.sql.types import StringType


spark = SparkSession.builder.getOrCreate()

df = spark.read.csv(r'C:\Users\User\.cache\kagglehub\datasets\gregorut\videogamesales\versions\2\vgsales.csv', header=True)
df = df.na.drop("any")
df = df.withColumn("Platform", col("Platform").cast(StringType()))
df = df.withColumn("Rank", col("Rank").cast("integer"))
df = df.withColumn("Year", col("Year").cast("string"))
df = df.withColumn("NA_Sales", col("NA_Sales").cast("float"))
df = df.withColumn("EU_Sales", col("EU_Sales").cast("float"))
df = df.withColumn("JP_Sales", col("JP_Sales").cast("float"))
df = df.withColumn("Other_Sales", col("Other_Sales").cast("float"))
df = df.withColumn("Global_Sales", col("Global_Sales").cast("float"))
df.show(10)

+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports| Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform| Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing| Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports| Nintendo|   15.75|   11.01|    3.28|       2.96|        33.0|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing| Nintendo|   11.27|    8.89|   10.22|        1.0|       31.37|
|   6|              Tetris|      GB|1989|      Puzzle| Nintendo|

In [6]:
# подажи по годам
df = df.withColumn("TotalByYear", sum("Global_Sales").over(Window.partitionBy("Year")))

# продажи по регионам с группировкой по годам и платформе
df = df.withColumn("TotalNA", sum("NA_Sales").over(Window.partitionBy("Year", "Platform")))
df = df.withColumn("TotalJP", sum("JP_Sales").over(Window.partitionBy("Year", "Platform")))
df = df.withColumn("TotalEU", sum("EU_Sales").over(Window.partitionBy("Year", "Platform")))
df = df.withColumn("TotalOtherRegions", sum("Other_Sales").over(Window.partitionBy("Year", "Platform")))
df.select("TotalByYear", "TotalNA", "TotalEU", "TotalOtherRegions").show(10, truncate=False)

+------------------+------------------+------------------+-------------------+
|TotalByYear       |TotalNA           |TotalEU           |TotalOtherRegions  |
+------------------+------------------+------------------+-------------------+
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|11.379999861121178|10.590000033378601|0.6699999906122684|0.11999999918043613|
|35.77000015974045 |33.39999992400408 |1.95999997481

In [9]:
df = df.withColumn('PartGamesNA', col('TotalNA') / sum('NA_Sales').over(Window.partitionBy("Year", "Platform", "Name")))
df = df.withColumn('PartGamesJP', col('TotalJP') / sum('JP_Sales').over(Window.partitionBy("Year", "Platform", "Name")))
df = df.withColumn('PartGamesEU', col('TotalEU') / sum('EU_Sales').over(Window.partitionBy("Year", "Platform", "Name")))
df = df.withColumn('PartGamesOtherRegions', col('TotalOtherRegions') / sum('Other_Sales').over(Window.partitionBy("Year", "Platform", "Name")))
df.show(10, truncate=False)

+----+------------------+--------+----+--------+------------------+--------+--------+--------+-----------+------------+------------------+------------------+-------+------------------+-------------------+------------------+-----------+------------------+---------------------+
|Rank|Name              |Platform|Year|Genre   |Publisher         |NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|TotalByYear       |TotalNA           |TotalJP|TotalEU           |TotalOtherRegions  |PartGamesNA       |PartGamesJP|PartGamesEU       |PartGamesOtherRegions|
+----+------------------+--------+----+--------+------------------+--------+--------+--------+-----------+------------+------------------+------------------+-------+------------------+-------------------+------------------+-----------+------------------+---------------------+
|259 |Asteroids         |2600    |1980|Shooter |Atari             |4.0     |0.26    |0.0     |0.05       |4.31        |11.379999861121178|10.590000033378601|0.0    |0.66

In [11]:
df = df.withColumn('PublisherProducing', col('Global_Sales') / sum("Global_Sales").over(Window.partitionBy("Publisher")))
df = df.withColumn('TatalGameSale', sum("Global_Sales").over(Window.partitionBy("Name")))
df.show(10, truncate=False)

+-----+----------------------+--------+----+------------+---------------+--------+--------+--------+-----------+------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+---------------------+-------------------+
|Rank |Name                  |Platform|Year|Genre       |Publisher      |NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|TotalByYear       |TotalNA           |TotalJP           |TotalEU           |TotalOtherRegions |PartGamesNA       |PartGamesJP       |PartGamesEU       |PartGamesOtherRegions|PublisherProducing   |TatalGameSale      |
+-----+----------------------+--------+----+------------+---------------+--------+--------+--------+-----------+------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+-----------------

In [ ]:
df.write.parquet("result.parquet", mode="overwrite")